In [1]:
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
# Cross-attention fusion for mapped drugs only

class CrossAttentionFusion(nn.Module):
    def __init__(self, in_smiles: int, in_rdkit: int, in_protein: int, embed_dim: int = 256, num_heads: int = 4):
        super().__init__()
        self.smiles_proj = nn.Linear(in_smiles, embed_dim)
        self.rdkit_proj = nn.Linear(in_rdkit, embed_dim)
        self.protein_proj = nn.Linear(in_protein, embed_dim)

        self.fusion_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.normal_(self.fusion_token, std=0.02)

        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, e_smiles: torch.Tensor, e_rdkit: torch.Tensor, e_protein: torch.Tensor) -> torch.Tensor:
        e_smiles = self.smiles_proj(e_smiles)
        e_rdkit = self.rdkit_proj(e_rdkit)
        e_protein = self.protein_proj(e_protein)

        e_smiles = F.normalize(e_smiles, p=2, dim=-1)
        e_rdkit = F.normalize(e_rdkit, p=2, dim=-1)
        e_protein = F.normalize(e_protein, p=2, dim=-1)

        modalities = torch.stack([e_smiles, e_rdkit, e_protein], dim=1)
        token = self.fusion_token.expand(e_smiles.size(0), -1, -1)

        attn_out, _ = self.cross_attn(query=token, key=modalities, value=modalities)
        x = self.norm1(token + attn_out)
        x = self.norm2(x + self.ffn(x))
        return F.normalize(x.squeeze(1), p=2, dim=-1)


In [3]:
# Paths (adjust if needed)
protein_path = Path("drug_protein_embeddings.pt")
smiles_path = Path("chemberta_smiles_embeddings.pt")
rdkit_path = Path("drug_rdkit_embeddings.pkl")
output_path = Path("drug_fused_embeddings_mapped_only.pt")

if not protein_path.exists():
    raise FileNotFoundError(f"Missing file: {protein_path}")
if not smiles_path.exists():
    raise FileNotFoundError(f"Missing file: {smiles_path}")
if not rdkit_path.exists():
    raise FileNotFoundError(f"Missing file: {rdkit_path}")

protein_dict = torch.load(protein_path, map_location="cpu", weights_only=False)
smiles_dict = torch.load(smiles_path, map_location="cpu", weights_only=False)
with open(rdkit_path, "rb") as f:
    rdkit_dict = pickle.load(f)

common_keys = sorted([k for k in protein_dict.keys() if (k in smiles_dict and k in rdkit_dict)])
print(f"Mapped protein drugs kept: {len(common_keys)}")

def to_vec(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().reshape(-1)
    return torch.tensor(np.asarray(x, dtype=np.float32)).reshape(-1)

E_protein = torch.stack([to_vec(protein_dict[k]) for k in common_keys], dim=0)
E_smiles = torch.stack([to_vec(smiles_dict[k]) for k in common_keys], dim=0)
E_rdkit = torch.stack([to_vec(rdkit_dict[k]) for k in common_keys], dim=0)

print("Input shapes:", E_smiles.shape, E_rdkit.shape, E_protein.shape)


Mapped protein drugs kept: 547
Input shapes: torch.Size([547, 384]) torch.Size([547, 256]) torch.Size([547, 1280])


In [4]:
torch.manual_seed(42)

model = CrossAttentionFusion(
    in_smiles=E_smiles.shape[1],
    in_rdkit=E_rdkit.shape[1],
    in_protein=E_protein.shape[1],
    embed_dim=256,
    num_heads=4,
).eval()

with torch.no_grad():
    fused = model(E_smiles, E_rdkit, E_protein)

fused_dict = {k: fused[i].cpu() for i, k in enumerate(common_keys)}
torch.save(fused_dict, output_path)

print(f"Saved: {output_path}")
print(f"Fused embeddings: {len(fused_dict)} | shape per drug: {tuple(fused.shape[1:])}")


Saved: drug_fused_embeddings_mapped_only.pt
Fused embeddings: 547 | shape per drug: (256,)


## Embedding quality checks

In [11]:
# Build a dense matrix from the saved dict to avoid stale variables from previous runs
fused_mat = torch.stack([fused_dict[k] for k in common_keys], dim=0).float()

norms = torch.norm(fused_mat, dim=1)
print("Mean norm:", norms.mean().item())
print("Std norm:", norms.std().item())

Mean norm: 1.0
Std norm: 3.851689100997646e-08


In [12]:
from torch.nn.functional import cosine_similarity

num_samples = min(1000, fused_mat.size(0))
idx = torch.randperm(fused_mat.size(0))[:num_samples]

sims = []
for i in range(len(idx) - 1):
    sims.append(
        cosine_similarity(
            fused_mat[idx[i]].unsqueeze(0),
            fused_mat[idx[i + 1]].unsqueeze(0),
        ).item()
    )

print("Mean cosine:", float(np.mean(sims)))
print("Std cosine:", float(np.std(sims)))

Mean cosine: 0.8968915408784217
Std cosine: 0.022816265347338806


In [13]:
y = torch.load("Y_labels.pt", map_location="cpu", weights_only=False)
Y = torch.as_tensor(y).float()[: len(common_keys)]
N = len(common_keys)

print("Y shape:", tuple(Y.shape))

Y shape: (547, 963)


In [14]:
import random
from sklearn.metrics import roc_auc_score

scores = []
labels = []

for _ in range(50000):  # sample pairs
    i = random.randint(0, N - 1)
    j = random.randint(0, N - 1)

    if i == j:
        continue

    sim = cosine_similarity(
        fused_mat[i].unsqueeze(0),
        fused_mat[j].unsqueeze(0),
    ).item()

    overlap = (Y[i] * Y[j]).sum().item()
    label = 1 if overlap > 0 else 0

    scores.append(sim)
    labels.append(label)

if len(set(labels)) < 2:
    print("Sampled AUC: undefined (only one class in sampled labels)")
else:
    auc = roc_auc_score(labels, scores)
    print("Sampled AUC:", float(auc))

Sampled AUC: 0.5163788575311812


In [15]:
# Label-space diagnostics
label_counts = Y.sum(dim=1)
print("Avg labels per drug:", float(label_counts.mean().item()))
print("Median labels per drug:", float(label_counts.median().item()))
print("Min/Max labels per drug:", int(label_counts.min().item()), int(label_counts.max().item()))
print("Label density:", float(label_counts.mean().item() / Y.shape[1]))

# Fraction of random pairs with any overlap (important baseline for overlap metrics)
import random
pair_trials = 20000
overlap_hits = 0
for _ in range(pair_trials):
    i = random.randint(0, N - 1)
    j = random.randint(0, N - 1)
    if i == j:
        continue
    overlap_hits += int((Y[i] * Y[j]).sum().item() > 0)
print("P(random pair has overlap > 0):", float(overlap_hits / pair_trials))

Avg labels per drug: 55.77513885498047
Median labels per drug: 41.0
Min/Max labels per drug: 1 217
Label density: 0.057918108883676496
P(random pair has overlap > 0): 0.9319


In [16]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

sim_matrix = cos_sim(fused_mat.cpu().numpy())

k = 10
correct = 0

for i in range(N):
    neighbors = sim_matrix[i].argsort()[-k - 1 : -1]

    for j in neighbors:
        if (Y[i] * Y[j]).sum().item() > 0:
            correct += 1

print("Recall@10:", float(correct / (N * k)))

Recall@10: 0.9394881170018281


In [17]:
# Positive vs negative cosine diagnostics
import random

num_pairs = 50000
pos_sims = []
neg_sims = []

for _ in range(num_pairs):
    i = random.randint(0, N - 1)
    j = random.randint(0, N - 1)
    if i == j:
        continue

    sim = cosine_similarity(
        fused_mat[i].unsqueeze(0),
        fused_mat[j].unsqueeze(0),
    ).item()

    if (Y[i] * Y[j]).sum().item() > 0:
        pos_sims.append(sim)
    else:
        neg_sims.append(sim)

print("Positive pairs:", len(pos_sims), "Negative pairs:", len(neg_sims))
if pos_sims and neg_sims:
    print("Pos cosine mean/std:", float(np.mean(pos_sims)), float(np.std(pos_sims)))
    print("Neg cosine mean/std:", float(np.mean(neg_sims)), float(np.std(neg_sims)))
    print("Separation (pos-neg mean):", float(np.mean(pos_sims) - np.mean(neg_sims)))

Positive pairs: 46591 Negative pairs: 3335
Pos cosine mean/std: 0.89688059349555 0.022728700406598017
Neg cosine mean/std: 0.8963600999769242 0.021262603591185456
Separation (pos-neg mean): 0.0005204935186258197


In [18]:
# Random baseline for Recall@10 (same overlap criterion)
trials = 200
rand_scores = []

for _ in range(trials):
    correct_rand = 0
    for i in range(N):
        # sample k random neighbors excluding self
        pool = np.delete(np.arange(N), i)
        neighbors = np.random.choice(pool, size=k, replace=False)
        for j in neighbors:
            if (Y[i] * Y[j]).sum().item() > 0:
                correct_rand += 1
    rand_scores.append(correct_rand / (N * k))

print("Random Recall@10 mean/std:", float(np.mean(rand_scores)), float(np.std(rand_scores)))
print("Model Recall@10:", float(correct / (N * k)))

Random Recall@10 mean/std: 0.9332175502742232 0.002910785623791629
Model Recall@10: 0.9394881170018281


In [19]:
# MAP@10 and NDCG@10 using overlap count as graded relevance
K = 10
map_scores = []
ndcg_scores = []

for i in range(N):
    sims = sim_matrix[i].copy()
    sims[i] = -1e9  # exclude self
    topk = np.argsort(sims)[-K:][::-1]

    rel = np.array([(Y[i] * Y[j]).sum().item() for j in topk], dtype=np.float32)
    bin_rel = (rel > 0).astype(np.float32)

    # AP@K
    if bin_rel.sum() > 0:
        csum = np.cumsum(bin_rel)
        precision_at_k = csum / (np.arange(K) + 1)
        ap = float((precision_at_k * bin_rel).sum() / bin_rel.sum())
    else:
        ap = 0.0
    map_scores.append(ap)

    # NDCG@K (graded)
    gains = (2.0 ** rel - 1.0)
    discounts = np.log2(np.arange(2, K + 2))
    dcg = float((gains / discounts).sum())

    ideal_rel = np.sort(rel)[::-1]
    ideal_gains = (2.0 ** ideal_rel - 1.0)
    idcg = float((ideal_gains / discounts).sum())
    ndcg = (dcg / idcg) if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)

print("MAP@10:", float(np.mean(map_scores)))
print("NDCG@10:", float(np.mean(ndcg_scores)))

MAP@10: 0.9475988778780521
NDCG@10: 0.5414336988560393
